# Pre-Processing the data

In [ ]:
import os
import zipfile
import random
import shutil
import numpy as np

# 1. Baixa o dataset oficial (Hospedado pela Microsoft)
!wget --no-check-certificate \
    "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip" \
    -O /tmp/cats_and_dogs.zip

# 2. Extrai o arquivo principal
zip_ref = zipfile.ZipFile('/tmp/cats_and_dogs.zip', 'r')
zip_ref.extractall('/tmp')
zip_ref.close()

# 3. Recria a estrutura de pastas idêntica à do "filtered" original
base_dir = '/tmp/cats_and_dogs_filtered'
for split in ['train', 'validation']: #aqui temos as imagens divididas em treino e validação
    for classe in ['cats', 'dogs']: #em cada pasta de treino e validação temos uma pasta de gatos e cachorros
        os.makedirs(os.path.join(base_dir, split, classe), exist_ok=True)

# 4. Função para embaralhar e separar um limite de fotos (1000 Treino / 500 Validação)
def split_data(SOURCE, TRAINING, VALIDATION):
    files = []
    for filename in os.listdir(SOURCE):
        file = os.path.join(SOURCE, filename)
        # Ignora arquivos corrompidos (comum em datasets reais)
        if os.path.getsize(file) > 0 and filename.endswith('.jpg'):
            files.append(filename)

    # Embaralha as imagens
    shuffled_set = random.sample(files, len(files))

    # Pega apenas 1500 imagens de cada classe para ficar idêntico ao tutorial antigo
    for i, filename in enumerate(shuffled_set[:1500]):
        src = os.path.join(SOURCE, filename)
        if i < 1000:
            shutil.copyfile(src, os.path.join(TRAINING, filename))
        else:
            shutil.copyfile(src, os.path.join(VALIDATION, filename))

# Caminhos de origem (como a Microsoft nomeia)
CAT_SOURCE_DIR = "/tmp/PetImages/Cat/"
DOG_SOURCE_DIR = "/tmp/PetImages/Dog/"

# Caminhos de destino (como o seu código espera)
TRAIN_CATS_DIR = os.path.join(base_dir, "train/cats/")
VAL_CATS_DIR = os.path.join(base_dir, "validation/cats/")
TRAIN_DOGS_DIR = os.path.join(base_dir, "train/dogs/")
VAL_DOGS_DIR = os.path.join(base_dir, "validation/dogs/")

split_data(CAT_SOURCE_DIR, TRAIN_CATS_DIR, VAL_CATS_DIR)
split_data(DOG_SOURCE_DIR, TRAIN_DOGS_DIR, VAL_DOGS_DIR)



--2026-06-08 03:54:53--  https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
Resolving download.microsoft.com (download.microsoft.com)... 23.200.181.207, 2600:1413:5000:688::317f, 2600:1413:5000:693::317f
Connecting to download.microsoft.com (download.microsoft.com)|23.200.181.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 824887076 (787M) [application/octet-stream]
Saving to: ‘/tmp/cats_and_dogs.zip’

/tmp/cats_and_dogs. 100%[===================>] 786.67M  73.1MB/s    in 10s     

2026-06-08 03:55:03 (75.7 MB/s) - ‘/tmp/cats_and_dogs.zip’ saved [824887076/824887076]



In [ ]:
import torch
from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader

base_dir = '/tmp/cats_and_dogs_filtered'
train_dir = f"{base_dir}/train"
val_dir = f"{base_dir}/validation"

prep_transform = T.Compose(
    [
        T.ToTensor(),
        T.Normalize(
            (0.4914, 0.4822, 0.4465),
            (0.2470, 0.2435, 0.2616)
        ),
        T.Resize(size=(30,30))
    ]
)

# O ImageFolder junta automaticamente as subpastas e cria as classes (0 e 1)
train_dataset = datasets.ImageFolder(root=train_dir, transform=prep_transform) #é uma matriz, na qual cada elemento dela é uma tupla, na qual o primeiro elemento é um Tensor e o Segundo um número inteiro
val_dataset = datasets.ImageFolder(root=val_dir, transform=prep_transform)

train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
validate_dataloader = DataLoader(val_dataset,batch_size=32, shuffle=True)
print(train_dataloader)


# Building the NN

In [ ]:
import torch
from torch import nn

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.convlayer = nn.Sequential( #padding diz que a imagem perderá as bordas pois a matriz de kernel não estará centrada nas pontas da imagem (matriz)
        nn.Conv2d(in_channels=3, out_channels=32, kernel_size= (3,3), padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2)), #diminui a matriz para uma matriz de 2x2, pega o maior elemento de cada quadrante para tentar manter a informação
        nn.Conv2d(in_channels=32, out_channels=64, kernel_size= (3,3), padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2))
    )

    self.linear = nn.Sequential(
        nn.Linear(3136,256),
        nn.ReLU(),
        nn.Linear(256,2)
    )

  def forward(self,x):
    v = self.convlayer(x)
    v = self.flatten(v)
    return self.linear(v)

In [ ]:
conv_model = CNN().to(device)
conv_optim = torch.optim.SGD(conv_model.parameters(), lr = 0.001)
lossfunc = nn.CrossEntropyLoss()


In [ ]:
def train(model, dataloader, loss_func, optimizer):
  model.train()
  cumloss = 0.0
  acertos = 0

  for imgs, labels in dataloader:
    imgs, labels = imgs.to(device), labels.to(device)
    pred = model(imgs)

    loss = loss_func(pred, labels)
    #zera os gradientes acumulados
    optimizer.zero_grad()
    #computa os gradientes
    loss.backward()
    # anda no sentido de diminuir o gradiente, ou seja, reduz o erro local
    optimizer.step()

    cumloss += loss.item()

  return cumloss/len(dataloader)

def test(model, dataloader, lossfunc):
  cumloss = 0.0
  acertos = 0
  total_imagens = 0
  model.eval()
  with torch.no_grad():
    for imgs, labels in dataloader:
      imgs, labels = imgs.to(device), labels.to(device)
      pred = model(imgs)
      loss = lossfunc(pred, labels)

      cumloss += loss.item()
      decided = torch.argmax(pred, 1)
      acertos += (decided == labels).sum().item()
      total_imagens += labels.size(0)
      acuracia = acertos/total_imagens

    return cumloss/len(dataloader), acuracia


In [ ]:
epochs = 21
loss_arr = []
for t in range(epochs):
  train_loss = train(conv_model,train_dataloader,lossfunc,conv_optim)
  loss_arr.append(train_loss)
  print(f'Epoch {t} Loss: {train_loss}')

Epoch 0 Loss: 0.6877324894673562
Epoch 1 Loss: 0.6830778878203706
Epoch 2 Loss: 0.6783786772648034
Epoch 3 Loss: 0.6736732862588298
Epoch 4 Loss: 0.6683701524844748
Epoch 5 Loss: 0.6629326328032279
Epoch 6 Loss: 0.6574886813329134
Epoch 7 Loss: 0.6512616025230099
Epoch 8 Loss: 0.6455294524314087
Epoch 9 Loss: 0.6398205341975813
Epoch 10 Loss: 0.6348874334655056
Epoch 11 Loss: 0.6294872559219427
Epoch 12 Loss: 0.6250933832515871
Epoch 13 Loss: 0.6207568542116639
Epoch 14 Loss: 0.6162788524276259
Epoch 15 Loss: 0.6124561783895327
Epoch 16 Loss: 0.6089099823911756
Epoch 17 Loss: 0.6050581983748199
Epoch 18 Loss: 0.6015483196071117
Epoch 19 Loss: 0.5984505079901976
Epoch 20 Loss: 0.5949371683632019


In [ ]:
validate, acuraria = test(conv_model,validate_dataloader,lossfunc)
print(f'Loss: {validate} Accuracy: {acuraria}')

Loss: 0.5984708554658693 Accuracy: 0.679126213592233
